# PARC2026 02: Dataset Prepare
最初に3 EpisodeのMini Datasetを通し、成功後に800 Episodeへ展開します。Miniでは本番Dataset Manifestを昇格しません。

In [ ]:
%cd /content/Physical_ai
ARTIFACTS = '/content/drive/MyDrive/PARC2026/40_experiments/datasets'
FULL_SELECTION = f'{ARTIFACTS}/libero_plus_selection_v001.json'
MINI_SELECTION = f'{ARTIFACTS}/mini_selection_v001.json'
!mkdir -p {ARTIFACTS} /content/work/mini_source /content/work/mini_rlds
!python -m src.data.build_mini_selection --selection {FULL_SELECTION} --train-count 2 --validation-count 1 --output {MINI_SELECTION}

In [ ]:
PLAN = f'{ARTIFACTS}/mini_download_plan.json'
!python -m src.data.build_episode_download_plan --selection {MINI_SELECTION} --include-videos --output {PLAN}
!python -m src.data.download_selected_episodes --plan {PLAN} --output-dir /content/work/mini_source

In [ ]:
import os
os.environ['PROJECT_ROOT'] = '/content/Physical_ai'
os.environ['OPENVLA_ROOT'] = '/content/openvla-oft'
os.environ['SOURCE_ROOT'] = '/content/work/mini_source'
os.environ['SELECTION_FILE'] = MINI_SELECTION
os.environ['TFDS_ROOT'] = '/content/work/mini_rlds'
os.environ['BASE_CHECKPOINT'] = '/content/work/models/openvla_oft_plus_base'
os.environ['ARTIFACT_ROOT'] = f'{ARTIFACTS}/mini_e2e_v001'
os.environ['PARITY_EPISODES_PER_SPLIT'] = '1'
os.environ['COMPATIBILITY_SAMPLES_PER_SPLIT'] = '4'
os.environ['PROMOTE_MANIFEST'] = '0'
!bash training/openvla_oft_a100/scripts/prepare_stage_a_rlds.sh

Miniで`rlds_conversion_report.json`、`rlds_source_parity.json`、`openvla_rlds_compatibility.json`が全てPassした場合だけ、800 Episode Selectionで`PROMOTE_MANIFEST=1`として本番Manifestを昇格します。